# Proof of Concept

Calculate a simple forward pass from 3DGS to 3DGS

In [ ]:
from dataset import UCO3DDataset

dataset = UCO3DDataset(
    data_root="./data",
    split="train",
    image_height=800,
    image_width=800,
    box_crop_context=0.4,
    gaussian_splats_truncate_background=True,
    normalize_splats=True,
)

frame_data = dataset[100]
gaussian_splats = frame_data["splat_tensor"]

print(f"Sequence: {frame_data['sequence_name']}")
print(f"Frame Number: {frame_data['frame_number']}")
print(f"Gaussian splats means shape: {gaussian_splats.means.shape}")

In [18]:
input = input.flatten().unsqueeze(0)

input.shape

torch.Size([1, 622874])

In [ ]:
import torch.nn as nn

class POC_Network(nn.Module):
    def __init__(self, dim):
        super(POC_Network, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, x):
        x = self.model(x)
        return x

network = POC_Network(dim=input.shape[1])

In [ ]:
from torch.optim import AdamW
import torch.nn.functional as F

optim = AdamW(params=network.parameters(), betas=(0.9, 0.999),eps=1e-8, weight_decay=0.1)

num_epochs = 1000

input.to("mps")

for epoch in range(num_epochs):

    optim.zero_grad()

    recon = network(input)
    loss = F.mse_loss(input, recon)

    loss.backward()
    optim.step()

    if epoch % 50 == 0:
        print(f"step {epoch}: Loss = {loss.item():.6f}")

    

step 0: Loss = 5.985198
step 50: Loss = 4.115030
step 100: Loss = 2.295019
step 150: Loss = 1.021219
step 200: Loss = 0.407251
step 250: Loss = 0.164159
